In [1]:
import os
import sys

# Tell Spark to wait up to 1200 seconds for the Python worker
os.environ["SPARK_AUTH_SECRET"] = "secret"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("VSCodeConfig") \
    .config("spark.python.worker.reuse", "true") \
    .config("spark.network.timeout", "1200s") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()


## 11  Join two DataFrames using inner, left, right and full joins

In [6]:
# Sample DataFrames
df_emp=spark.createDataFrame(
    [(1,'Alice','HR'),(2,'Naveen','Engineering'),(3,'Praveen','Marketing'),(4,'Anand','IT')],
    ['emp_id','name','dept']
)

df_dept=spark.createDataFrame(
    [('HR','New York'),('Engineering','San Francisco'),('Finance','Chicago')],
    ['dept','location']
)

# Inner Join
inner_df=df_emp.join(df_dept, on="dept", how="inner")

# Left Join
left_df=df_emp.join(df_dept, on="dept", how="left")

# Right Join
right_df=df_emp.join(df_dept, on="dept", how="right")

# Full Outer Join
full_df=df_emp.join(df_dept, on="dept", how="outer")

In [8]:
right_df.show(5)
full_df.show(5)


+-----------+------+------+-------------+
|       dept|emp_id|  name|     location|
+-----------+------+------+-------------+
|         HR|     1| Alice|     New York|
|Engineering|     2|Naveen|San Francisco|
|    Finance|  NULL|  NULL|      Chicago|
+-----------+------+------+-------------+

+-----------+------+-------+-------------+
|       dept|emp_id|   name|     location|
+-----------+------+-------+-------------+
|Engineering|     2| Naveen|San Francisco|
|    Finance|  NULL|   NULL|      Chicago|
|         HR|     1|  Alice|     New York|
|         IT|     4|  Anand|         NULL|
|  Marketing|     3|Praveen|         NULL|
+-----------+------+-------+-------------+



## 12. Perfor Self join on a DataFrame

In [28]:
spark=SparkSession.builder.appName('Read_CSV').getOrCreate()
df=spark.read.option('header', True).option('inferSchema', True).csv("data/employee.csv")

df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- emp_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- join_date: timestamp (nullable = true)



In [29]:
#Perform a self join on a DataFrame
df_alias1=df.alias("e1")
df_alias2=df.alias("e2")

from pyspark.sql.functions import col

self_join_df=df_alias1.join(
    df_alias2,
    (col('e1.department') == col('e2.department')) & (col('e1.emp_id')<col('e2.emp_id'))
) \
    .select( col('e1.emp_id').alias('emp_id_1'),col('e1.emp_name').alias('emp_name_1'),col('e2.emp_id').alias('emp_id_2'), col('e2.emp_name').alias('emp_name_2'),col('e1.department')
    
    )

self_join_df.show(truncate=False)

+--------+------------------+--------+------------------+------------+
|emp_id_1|emp_name_1        |emp_id_2|emp_name_2        |department  |
+--------+------------------+--------+------------------+------------+
|1       |A. Kannan         |8       |Alon Fliess       | Engineering|
|1       |A. Kannan         |6       |Alex Mackman      | Engineering|
|1       |A. Kannan         |4       |Adity Gupta       | Engineering|
|1       |A. Kannan         |2       |Aaron M. Tenenbaum| Engineering|
|2       |Aaron M. Tenenbaum|8       |Alon Fliess       | Engineering|
|2       |Aaron M. Tenenbaum|6       |Alex Mackman      | Engineering|
|2       |Aaron M. Tenenbaum|4       |Adity Gupta       | Engineering|
|3       |Adam Smith        |9       |Ananth Grama      | Marketing  |
|4       |Adity Gupta       |8       |Alon Fliess       | Engineering|
|4       |Adity Gupta       |6       |Alex Mackman      | Engineering|
|5       |Alex lonescu      |10      |Andre Tost        | Sales      |
|5    

## 13 Use Window Function to calculate Rank, Dense_rank and row number



In [30]:
from pyspark.sql import Window
from pyspark.sql.functions import col, rank, dense_rank, row_number

window_spec=Window.partitionBy('department').orderBy(col('salary').desc())


df_ranked=(
    df.withColumn('rank', rank().over(window_spec)) \
        .withColumn('dense_rank', dense_rank().over(window_spec)) \
            .withColumn('row_number', row_number().over(window_spec))
)
df_ranked.show(truncate=False)


+------+------------------+------------+-------+-------------------+----+----------+----------+
|emp_id|emp_name          |department  |salary |join_date          |rank|dense_rank|row_number|
+------+------------------+------------+-------+-------------------+----+----------+----------+
|1     |A. Kannan         | Engineering|70000.0|2021-06-15 00:00:00|1   |1         |1         |
|2     |Aaron M. Tenenbaum| Engineering|70000.0|2021-06-01 00:00:00|1   |1         |2         |
|4     |Adity Gupta       | Engineering|60000.0|2021-03-17 00:00:00|3   |2         |3         |
|6     |Alex Mackman      | Engineering|40000.0|2021-01-19 00:00:00|4   |3         |4         |
|8     |Alon Fliess       | Engineering|35000.0|2021-12-05 00:00:00|5   |4         |5         |
|3     |Adam Smith        | Marketing  |76000.0|2021-07-19 00:00:00|1   |1         |1         |
|9     |Ananth Grama      | Marketing  |30000.0|2021-03-20 00:00:00|2   |2         |2         |
|5     |Alex lonescu      | Sales      |

## 14.  Find the Second Highest Salary

In [36]:
from pyspark.sql.functions import desc
df.select('salary').distinct().orderBy(desc('salary')).limit(2).offset(1).show()

+-------+
| salary|
+-------+
|70000.0|
+-------+



## 15. Calculate the running total using Window Functions

In [38]:
from pyspark.sql import Window
from pyspark.sql.functions import sum as _sum

window_spec=Window.partitionBy('Department').orderBy(col('salary').desc()) \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_running=df.withColumn('runningSum', _sum('salary').over(window_spec)) \
    .orderBy('department', desc('salary'))

df_running.show(truncate=False)

+------+------------------+------------+-------+-------------------+----------+
|emp_id|emp_name          |department  |salary |join_date          |runningSum|
+------+------------------+------------+-------+-------------------+----------+
|1     |A. Kannan         | Engineering|70000.0|2021-06-15 00:00:00|70000.0   |
|2     |Aaron M. Tenenbaum| Engineering|70000.0|2021-06-01 00:00:00|140000.0  |
|4     |Adity Gupta       | Engineering|60000.0|2021-03-17 00:00:00|200000.0  |
|6     |Alex Mackman      | Engineering|40000.0|2021-01-19 00:00:00|240000.0  |
|8     |Alon Fliess       | Engineering|35000.0|2021-12-05 00:00:00|275000.0  |
|3     |Adam Smith        | Marketing  |76000.0|2021-07-19 00:00:00|76000.0   |
|9     |Ananth Grama      | Marketing  |30000.0|2021-03-20 00:00:00|106000.0  |
|5     |Alex lonescu      | Sales      |50000.0|2021-02-18 00:00:00|50000.0   |
|7     |Alfred.V.Aho      | Sales      |45000.0|2021-11-25 00:00:00|95000.0   |
|10    |Andre Tost        | Sales      |

## 16. Pivot the DataFrame to get total salary by department

In [41]:
from pyspark.sql.functions import sum as _sum
pivot_df=df.groupBy('department').pivot('department').agg(_sum('salary').alias('Total_salary'))
pivot_df.fillna(0).orderBy('department').show(truncate=False)

+------------+------------+----------+--------+
|department  | Engineering| Marketing| Sales  |
+------------+------------+----------+--------+
| Engineering|275000.0    |0.0       |0.0     |
| Marketing  |0.0         |106000.0  |0.0     |
| Sales      |0.0         |0.0       |120000.0|
+------------+------------+----------+--------+



## 17. Find Employee who earn more than the average salary in their department

In [44]:
from pyspark.sql import Window
from pyspark.sql.functions import avg, col

# avg salary per department
w=Window.partitionBy('department')
df_with_avg=df.withColumn('avg_salary', avg('salary').over(w))
df_with_avg.show(truncate=False)

# Filter employees earning  more than department avg
result_df=df_with_avg.filter(col('salary')>col('avg_salary')) \
    .drop('avg_salary').orderBy('Department','salary', ascending=False)

result_df.show(truncate=False)


+------+------------------+------------+-------+-------------------+----------+
|emp_id|emp_name          |department  |salary |join_date          |avg_salary|
+------+------------------+------------+-------+-------------------+----------+
|1     |A. Kannan         | Engineering|70000.0|2021-06-15 00:00:00|55000.0   |
|2     |Aaron M. Tenenbaum| Engineering|70000.0|2021-06-01 00:00:00|55000.0   |
|4     |Adity Gupta       | Engineering|60000.0|2021-03-17 00:00:00|55000.0   |
|6     |Alex Mackman      | Engineering|40000.0|2021-01-19 00:00:00|55000.0   |
|8     |Alon Fliess       | Engineering|35000.0|2021-12-05 00:00:00|55000.0   |
|3     |Adam Smith        | Marketing  |76000.0|2021-07-19 00:00:00|53000.0   |
|9     |Ananth Grama      | Marketing  |30000.0|2021-03-20 00:00:00|53000.0   |
|5     |Alex lonescu      | Sales      |50000.0|2021-02-18 00:00:00|40000.0   |
|7     |Alfred.V.Aho      | Sales      |45000.0|2021-11-25 00:00:00|40000.0   |
|10    |Andre Tost        | Sales      |

## 18 Find the Month with Highest Total Salary paid

In [47]:
from pyspark.sql.functions import to_date, date_format, sum as _sum

df_with_month=df.withColumn('month',date_format(to_date('join_date'),'yyyy-MM'))
monthly_salary=df_with_month.groupBy('month').agg(_sum('salary').alias('total_salary'))
monthly_salary.orderBy(desc('total_salary')).show(truncate=False)
monthly_salary.orderBy(desc('total_salary')).limit(1).show(truncate=False)

+-------+------------+
|month  |total_salary|
+-------+------------+
|2021-06|140000.0    |
|2021-03|90000.0     |
|2021-07|76000.0     |
|2021-02|50000.0     |
|2021-11|45000.0     |
|2021-01|40000.0     |
|2021-12|35000.0     |
|2021-05|25000.0     |
+-------+------------+

+-------+------------+
|month  |total_salary|
+-------+------------+
|2021-06|140000.0    |
+-------+------------+



## 19 Find Duplicate records based on all columns

In [49]:
from pyspark.sql.functions import count
dup_df=df.groupBy(df.columns).agg(count('*').alias('count')) \
    .filter('count>1').drop('count')

dup_df.show(truncate=False)

+------+--------+----------+------+---------+
|emp_id|emp_name|department|salary|join_date|
+------+--------+----------+------+---------+
+------+--------+----------+------+---------+



## 20. Find the top earning employee in each department.

In [51]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number

w=Window.partitionBy('department').orderBy(col('salary').desc())

df_withrank=df.withColumn('rnk', row_number().over(w))
top_empDept=df_withrank.filter(col('rnk')==1).drop('rnk')
top_empDept.show(truncate=False)

+------+------------+------------+-------+-------------------+
|emp_id|emp_name    |department  |salary |join_date          |
+------+------------+------------+-------+-------------------+
|1     |A. Kannan   | Engineering|70000.0|2021-06-15 00:00:00|
|3     |Adam Smith  | Marketing  |76000.0|2021-07-19 00:00:00|
|5     |Alex lonescu| Sales      |50000.0|2021-02-18 00:00:00|
+------+------------+------------+-------+-------------------+



## 21. Find Employees who have not joined inthe given year

In [56]:
from pyspark.sql.functions import col, year
filter_year=2022
df_not_joined=df.filter(year(col('join_date')) !=filter_year)
df_not_joined.show(truncate=False)

+------+------------------+------------+-------+-------------------+
|emp_id|emp_name          |department  |salary |join_date          |
+------+------------------+------------+-------+-------------------+
|1     |A. Kannan         | Engineering|70000.0|2021-06-15 00:00:00|
|2     |Aaron M. Tenenbaum| Engineering|70000.0|2021-06-01 00:00:00|
|3     |Adam Smith        | Marketing  |76000.0|2021-07-19 00:00:00|
|4     |Adity Gupta       | Engineering|60000.0|2021-03-17 00:00:00|
|5     |Alex lonescu      | Sales      |50000.0|2021-02-18 00:00:00|
|6     |Alex Mackman      | Engineering|40000.0|2021-01-19 00:00:00|
|7     |Alfred.V.Aho      | Sales      |45000.0|2021-11-25 00:00:00|
|8     |Alon Fliess       | Engineering|35000.0|2021-12-05 00:00:00|
|9     |Ananth Grama      | Marketing  |30000.0|2021-03-20 00:00:00|
|10    |Andre Tost        | Sales      |25000.0|2021-05-11 00:00:00|
+------+------------------+------------+-------+-------------------+



## 22. Find department that have more than 3 employee

In [60]:
from pyspark.sql.functions import count
df.groupBy('department').agg(count('emp_id').alias('emp_count')) \
    .filter(col('emp_count')>3) \
        .orderBy('emp_count', ascending=False).show(truncate=False)

+------------+---------+
|department  |emp_count|
+------------+---------+
| Engineering|5        |
+------------+---------+



## 23. Find the Longest employee name

In [63]:
from pyspark.sql.functions import length, desc
df.select('emp_name', length(col('emp_name')).alias('name_length')) \
    .orderBy(desc('name_length')).limit(1).show(truncate=False)

+------------------+-----------+
|emp_name          |name_length|
+------------------+-----------+
|Aaron M. Tenenbaum|18         |
+------------------+-----------+



## 24. Add a new column  'Year of Joining' from 'join_date'

In [64]:
from pyspark.sql.functions import year
df_withYear=df.withColumn('year_of_joining', year(col('join_date')))
df_withYear.show(truncate=False)

+------+------------------+------------+-------+-------------------+---------------+
|emp_id|emp_name          |department  |salary |join_date          |year_of_joining|
+------+------------------+------------+-------+-------------------+---------------+
|1     |A. Kannan         | Engineering|70000.0|2021-06-15 00:00:00|2021           |
|2     |Aaron M. Tenenbaum| Engineering|70000.0|2021-06-01 00:00:00|2021           |
|3     |Adam Smith        | Marketing  |76000.0|2021-07-19 00:00:00|2021           |
|4     |Adity Gupta       | Engineering|60000.0|2021-03-17 00:00:00|2021           |
|5     |Alex lonescu      | Sales      |50000.0|2021-02-18 00:00:00|2021           |
|6     |Alex Mackman      | Engineering|40000.0|2021-01-19 00:00:00|2021           |
|7     |Alfred.V.Aho      | Sales      |45000.0|2021-11-25 00:00:00|2021           |
|8     |Alon Fliess       | Engineering|35000.0|2021-12-05 00:00:00|2021           |
|9     |Ananth Grama      | Marketing  |30000.0|2021-03-20 00:00:

## 25. Find pairs of employees who are in the same department

In [66]:
e1=df.alias('e1')
e2=df.alias('e2')

pairs_df=e1.join(e2, (col('e1.department')==col('e2.department')) & (col('e1.emp_id') < col('e2.emp_id')),'inner') \
    .select(col('e1.department').alias('department'), col('e1.emp_name').alias('emp1'), col('e2.emp_name').alias('emp2')) \
        .orderBy('department','emp1', 'emp2')

pairs_df.show(truncate=False)

+------------+------------------+------------------+
|department  |emp1              |emp2              |
+------------+------------------+------------------+
| Engineering|A. Kannan         |Aaron M. Tenenbaum|
| Engineering|A. Kannan         |Adity Gupta       |
| Engineering|A. Kannan         |Alex Mackman      |
| Engineering|A. Kannan         |Alon Fliess       |
| Engineering|Aaron M. Tenenbaum|Adity Gupta       |
| Engineering|Aaron M. Tenenbaum|Alex Mackman      |
| Engineering|Aaron M. Tenenbaum|Alon Fliess       |
| Engineering|Adity Gupta       |Alex Mackman      |
| Engineering|Adity Gupta       |Alon Fliess       |
| Engineering|Alex Mackman      |Alon Fliess       |
| Marketing  |Adam Smith        |Ananth Grama      |
| Sales      |Alex lonescu      |Alfred.V.Aho      |
| Sales      |Alex lonescu      |Andre Tost        |
| Sales      |Alfred.V.Aho      |Andre Tost        |
+------------+------------------+------------------+

